# Chapter 2 Lab — Part A: Byte-Pair Encoding (BPE) and Out-of-Vocabulary Words

**Estimated time:** 15 minutes · **Read-and-run.** Pure-Python mini-BPE, standard library only, no API key, no GPU.

Part A builds a BPE tokenizer from a four-line corpus and shows what happens to a word the
tokenizer has never seen. **Part B** (`chapter-02b-lab.ipynb`) contrasts BPE with a
SentencePiece-style unigram tokenizer and turns token counts into context-window and cost arithmetic.

In [ ]:
import re, collections
corpus = ['low lower lowest', 'newer wider', 'new news newest', 'the network learns']

def get_vocab(corpus):
    v = collections.Counter()
    for line in corpus:
        for w in line.split():
            v[' '.join(list(w)) + ' </w>'] += 1
    return v

def merges_step(vocab):
    pairs = collections.Counter()
    for word, freq in vocab.items():
        sym = word.split()
        for i in range(len(sym)-1):
            pairs[(sym[i], sym[i+1])] += freq
    if not pairs: return None, vocab
    best = max(pairs, key=pairs.get)
    bigram = re.escape(' '.join(best)); patt = re.compile(r'(?<!\S)'+bigram+r'(?!\S)')
    vocab = {patt.sub(''.join(best), w): f for w,f in vocab.items()}
    return best, vocab

In [ ]:
vocab = get_vocab(corpus)
merges = []
for _ in range(12):
    best, vocab = merges_step(vocab)
    if best is None: break
    merges.append(best)
print('learned merges (in order):')
for a, b in merges: print('  ', a, '+', b)

## Encode a word, including an out-of-domain one

In [ ]:
def bpe_encode(word, merges):
    sym = list(word) + ['</w>']
    for a,b in merges:
        i=0; merged=[]
        while i < len(sym):
            if i < len(sym)-1 and sym[i]==a and sym[i+1]==b:
                merged.append(a+b); i+=2
            else: merged.append(sym[i]); i+=1
        sym = merged
    return sym

for w in ['newer', 'lowest', 'transformer']:  # 'transformer' is out-of-domain here
    toks = bpe_encode(w, merges)
    print(f'{w:12s} -> {len(toks):2d} tokens  {toks}')
print('\nThe OOD word falls back to many short/character tokens - it is never <UNK>.')

## Exercise (5 minutes)
1. Add the line `'the transformer learns attention'` to `corpus`, re-run, and watch how many tokens `transformer` needs now.
2. Change the merge budget from 12 to 4. Which words get longer? That is the vocabulary-size trade-off in one line.

In [ ]:
# OPTIONAL: a real production BPE tokenizer, only if installed (pip install tiktoken).
try:
    import tiktoken
    enc = tiktoken.get_encoding('cl100k_base')
    for s in ['transformer', 'antidisestablishmentarianism', 'GPU kernels', 'NVIDIA NeMo']:
        print(f'{s:35s} -> {len(enc.encode(s))} tokens')
except Exception as e:
    print('tiktoken not installed (optional):', type(e).__name__)

## Key takeaway
Subword BPE keeps a bounded vocabulary while still representing unseen words (as more,
smaller tokens) instead of emitting `<UNK>`. Token count — not word count — drives context
usage and cost. Continue with **Part B** for the SentencePiece comparison and the context-window arithmetic.